In [ ]:
# Module A — In-Context Techniques (Reasoning Without Training)
# ---------------------------------------------------------------
# This notebook explores three light experiments:
# 1. Alignment-first prompting
# 2. Cross-lingual self-consistency
# 3. Translation ablation
# ---------------------------------------------------------------

In [ ]:
FULL_MODE = False  # set to True if you have transformers locally
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"  # any small HF model
LANGS = ["en", "it", "es", "de"]

try:
    if FULL_MODE:
        from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
        gen = pipeline("text-generation", model=MODEL_NAME)
        print(f"Loaded model {MODEL_NAME}")
    else:
        gen = None
except Exception:
    gen = None
    print("Transformers not available, using fallback mode.")


In [ ]:
# Alignment-First Prompt Template

alignment_first_card = """
## Alignment-First Reasoning Card

1. Abstract the problem in a language-agnostic schema (variables, relations, constraints).
2. Produce a short plan valid across languages.
3. Realise the reasoning and final answer in the target language.
4. Output format:
   answer: <final short answer>
   rationale: <concise reasoning steps>
"""

print(alignment_first_card)


In [ ]:
# Minimal Prompt Function

import random

def mock_model(prompt: str, lang: str) -> str:
    """Fallback generator — produces deterministic structured text."""
    random.seed(hash(prompt + lang) % (2**32))
    answer = {"en": "21", "it": "21", "es": "21"}[lang]
    rationale = {
        "en": "If Ana has twice as many apples as Ben (2x = 42), Ben has 21.",
        "it": "Se Ana ha il doppio delle mele di Ben (2x = 42), Ben ne ha 21.",
        "es": "Si Ana tiene el doble de manzanas que Ben (2x = 42), Ben tiene 21."
    }[lang]
    schema = "x + 2x = 42"
    return f"answer: {answer}\nrationale: {rationale}\nschema: {schema}"


def model_fn(prompt, lang):
    if gen is None:
        return mock_model(prompt, lang)
    else:
        return gen(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"]


In [ ]:
# Cross-Lingual Self-Consistency
import re
from collections import Counter

def extract_answer(text):
    m = re.search(r"answer\s*:\s*(.+)", text, re.I)
    return m.group(1).strip() if m else None

def cross_lingual_self_consistency(question, langs=LANGS):
    outputs, answers = {}, {}
    for L in langs:
        prompt = f"{alignment_first_card}\n[LANGUAGE: {L}]\nQuestion: {question}"
        out = model_fn(prompt, L)
        outputs[L] = out
        answers[L] = extract_answer(out)
    maj = Counter(answers.values()).most_common(1)[0][0]
    agreement = sum(1 for a in answers.values() if a == maj) / len(langs)
    return outputs, answers, maj, agreement

question = "Ana has twice as many apples as Ben, and together they have 42 apples. How many does Ben have?"
outs, ans, maj, agree = cross_lingual_self_consistency(question)

print("Majority answer:", maj)
print("Agreement rate:", round(agree * 100, 1), "%")
ans


In [ ]:
# Translation Ablation
def translate_to_en(text, src):
    # dummy translator (identity function)
    return text

def ablation_native_vs_pivot(question_native, lang="it"):
    # Path A: native reasoning
    out_native = model_fn(f"{alignment_first_card}\n[LANGUAGE: {lang}]\nQuestion: {question_native}", lang)
    # Path B: translate → English → reason → (back-translate placeholder)
    q_en = translate_to_en(question_native, lang)
    out_pivot = model_fn(f"{alignment_first_card}\n[LANGUAGE: en]\nQuestion: {q_en}", "en")
    return {"native": out_native, "pivot_en": out_pivot}

abl = ablation_native_vs_pivot("Se Ana ha il doppio delle mele di Ben e insieme ne hanno 42, quante ne ha Ben?", "it")
for k,v in abl.items():
    print(f"\n--- {k.upper()} ---\n{v}")

In [ ]:
# Summary Report
import pandas as pd

def summary_table(answers_dict):
    rows = [{"Language": L, "Answer": a} for L,a in answers_dict.items()]
    df = pd.DataFrame(rows)
    print(df)
    print("\nAgreement Rate:", round(agree*100,1), "% — Majority:", maj)
    return df

df_summary = summary_table(ans)